# Class 6. Exam prep

# Imports

In [2]:
import pandas as pd
import numpy as np

from IPython.core.display import HTML #for the fancy table formatting
table_css = 'table {align:left;display:block} '
HTML('<style>{}</style>'.format(table_css))

# Exercise 1. 3 Consecutive numbers

Table: ```logs```

| Column Name | Type    |
|-------------|---------|
| id          | int     |
| num         | varchar |

In SQL, id is the primary key for this table.
id is an autoincrement column starting from 1.
 

Find all numbers that appear at least three times consecutively.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Logs table:

| id | num |
|----|-----|
| 1  | 1   |
| 2  | 1   |
| 3  | 1   |
| 4  | 2   |
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |

Output: 

| ConsecutiveNums |
|-----------------|
| 1               |

Explanation: 1 is the only number that appears consecutively for at least three times.

In [31]:
logs = pd.DataFrame({'id' : range(1, 8), 'num' : [1,1,1,2,1,2,2]})
print(logs)



def consec(logs: pd.DataFrame) -> pd.DataFrame:
    if len(logs) < 3:
        return pd.DataFrame(columns='ConsecutiveNums')

    def func(x: pd.Series) -> None | int:
        # print(x, "\n\n\n")
        
        if (x['id'] not in {1, len(logs)}) and (logs.iloc[x['id']-2, 1] == x['num'] == logs.iloc[x['id'], 1]):
            return x['num']
            

    df = logs.apply(func, axis=1)
    df.dropna(inplace=True)
    df = df.astype(int)
    df = pd.DataFrame(data=df.tolist(), columns=['ConsecutiveNums'])
    return df

out = consec(logs)
print(out)

   id  num
0   1    1
1   2    1
2   3    1
3   4    2
4   5    1
5   6    2
6   7    2
   ConsecutiveNums
0                1


# Exercise 2. 1D particle simulation

Your goal in this task is to create a very simplified particle simulation. We have some array of size ```M``` representing a tube with the particles.
Denote each cell of the tube **with the particle** as **1** and each cell of the tube **without the particle** as **0**. Initially the particles are\
distributed randomly. To be precise, at the beginning for each cell has a particle with some probability ```p```. We simulate the particle movement for\
```T``` periods. At each period the particle can either move n cells to the left if its speed is -n, stay in place if its speed is 0 and move n cells to the right if\
its speed is +n. Initially particle speeds are distributed uniformly from ```-M``` to ```M```. With each iteration each particle slows down by a constant ```s```\
until the speed is 0. When a particle encounters the border, it should bounce off of it meaning that the sign of its speed value should change.\
When 2 or more particles are in the same cell, store it the same way you would store a single particle.

Create a function, which takes 4 arguments:\
```M: int``` - number of cells in the tube\
```p: float``` - probability of the particle appearing in the tube initially (optional, set default as 0.5)\
```s: int``` - the constant by which the speed of every particle slows on each iteration\
```T: int``` - number of periods of simulation

Name the function ```sim```\
Return the tube condition after simulation

Example:

Input:
    M = 3\
    p = .5\
    s = 1\
    T = 2

initial array:\
[1, 0, 1]

initial speeds:\
[2, -1]

after iteration 1:

    positions' array:
    [1, 0, 1] -> [0, 1, 1]

    Explanation: the particle in position 1 moved 2 cells to the right as its speed was 2, the particle at the end moved 1 cell to the left as its speed is -1

    new speeds:
    [1, 0]

after iteration 2:

    [0, 1, 1] -> [0, 1, 0]
    Explanation: the particle at the middle has a speed 0 so it doesn't move anywhere, the particle to the left has a speed 1 but because it can't move
    to the right anymore, it bounced off of the wall of the tube and moved 1 cell to the left

Output:

    [0, 1, 0]

In [7]:
def sim(M: int, p: float = 0.5, s: int = 1, T: int = 1) -> np.ndarray:
    
    np.random.seed(42)

    arr = np.random.binomial(1, p, M)
    print(f'init arr {arr}')
    coords = np.where(arr == 1)[0]
    vs = np.random.randint(-M, M + 1, coords.shape[0])
    print(f'init speeds {vs}')
    
    # Begin simulation (same as above)
    for t in range(T):
        # Update each particle's position
        for i in range(len(coords)):
            new_pos = coords[i] + vs[i]
            
            while new_pos < 0 or new_pos >= M:
                if new_pos < 0:
                    new_pos = -new_pos
                    vs[i] = -vs[i]
                elif new_pos >= M:
                    new_pos = 2 * (M - 1) - new_pos
                    vs[i] = -vs[i]
            
            coords[i] = new_pos
        
        # Update velocities (slow down)
        for i in range(len(vs)):
            if vs[i] > 0:
                vs[i] = max(0, vs[i] - s)
            elif vs[i] < 0:
                vs[i] = min(0, vs[i] + s)
        
        # Handle collisions
        unique_coords = np.unique(coords)
        if len(unique_coords) < len(coords):
            coords = unique_coords
            vs = vs[:len(unique_coords)]
    
    # Create final tube array
    result = np.zeros(M, dtype=int)
    result[coords] = 1
    
    return result

print(sim(3, .5, 1, 2))  # Should output [0, 1, 0]

init arr [0 1 1]
init speeds [1 1]
[0 1 1]


# Exercise 3. Customers who never order

Table: ```customers```

| Column Name | Type    |
|-------------|---------|
| id          | int     |
| name        | varchar |

id is the primary key (column with unique values) for this table.\
Each row of this table indicates the ID and name of a customer.
 

Table: ```Orders```


| Column Name | Type |
|-------------|------|
| id          | int  |
| customerId  | int  |

id is the primary key (column with unique values) for this table.\
customerId is a foreign key (reference columns) of the ID from the Customers table.\
Each row of this table indicates the ID of an order and the ID of the customer who ordered it.
 

Write a solution to find all customers who never order anything.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Customers table:

| id | name  |
|----|-------|
| 1  | Joe   |
| 2  | Henry |
| 3  | Sam   |
| 4  | Max   |

Orders table:

| id | customerId |
|----|------------|
| 1  | 3          |
| 2  | 1          |

Output: 

| Customers |
|-----------|
| Henry     |
| Max       |


In [34]:
orders = pd.DataFrame({'id' : [1, 2], 'customerId' : [3, 1]})
customers = pd.DataFrame({'id' : [1,2,3,4], 'name' : ['Joe', 'Henry', 'Sam', 'Max']})

# print(f'{orders}\n\n\n{customers}')

def CWNO(customers: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    df = customers.merge(orders, left_on='id', right_on='customerId', how='inner')

    # def mask_func(x):
    #     if x not in list(df['name']):
    #         return True
    #     else:
    #         return False

    # print(df)
    
    customers['mask'] = customers['name'].apply(lambda x: True if x not in list(df['name']) else False)
    # print('\n\n\n', customers[~customers['name'].isin(df['name'])])
    out = customers[customers['mask'] == True]
    out = out[['name']]
    out = out.rename(columns={'name' : 'customers'})
    out = out.reset_index(drop=True)
    return out

CWNO(customers, orders)

,customers
0,Henry
1,Max


# Exercise 4. Reverse an integer

Given a signed 32-bit integer x, return x with its digits reversed. If reversing x causes the value to go outside the signed 32-bit integer range [-2^31, 2^31 - 1], then return 0.

Assume the environment does not allow you to store 64-bit integers (signed or unsigned).

 

Example 1:

Input: x = 123\
Output: 321

Example 2:

Input: x = -123\
Output: -321

Example 3:

Input: x = 120\
Output: 21
 

Constraints:

1. -2^31 <= x <= 2^31 - 1

In [62]:
def rev_int(x: int) -> int:
    sign = None
    x_str = list(str(x))
    # print(x_str)
    if x_str[0] == '-':
        sign = x_str[0]
        x_str = x_str[1:]
    x_str = x_str[::-1]
    x_str = ''.join(x_str)
    x_str = x_str.strip('0')
    border = 2 ** 31
    # border = 1 << 31
    # print(x_str)
    
    if sign is None:
        border -= 1
        
    border = str(border)
    # print(border, x_str)
    max_len = max(len(border), len(x_str))
    # print(max_len)
    
    if max_len != len(border):
        return 0
        
    elif len(border) > len(x_str):
        if sign is None:
            return int(x_str) 
        else:
            return int(sign + x_str)
    else:
        for num_border, num_x in zip(border, x_str):
            if num_x > num_border:
                return 0
                
        if sign is None:
            return int(x_str)
        else:
            return int(sign + x_str)



rev_int(1231230002)

2000321321